In [8]:
# ── Logistic Regression ──────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

### Data Loading and Basic Inspection

In [6]:
X, y = load_breast_cancer().data, load_breast_cancer().target

In [7]:
X.shape, y.shape

((569, 30), (569,))


### 2. Data Splitting

#### Note the `stratify = y` to ensure that the distribution of the target is uniform in both the training and testing sets 

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    stratify=y, 
                                                    shuffle=True, 
                                                    random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(455, 30) (114, 30) (455,) (114,)


### 3. Defining the Pipeline including scaler and estimator (model)

In [9]:
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=50, random_state=42))
])

### 4. Cross-validation, and Stratified KFold, again to keep label distribution uniform

In [11]:
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
cv_scores = cross_val_score(lr_pipe, X_train, y_train,
                            cv=skf, scoring="roc_auc")

print(f"Baseline CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

Baseline CV AUC: 0.996 ± 0.005


### 5. Hyperparameter Tuning using GridSearch

In [12]:
# Definig the parameter grid for LogisticRegression's 'C'with GridSearchCV
param_grid = {
    "model__C": [0.1, 0.2,.4, .6]
}

lr_search = GridSearchCV(
    lr_pipe, param_grid, cv=skf,
    scoring="roc_auc", n_jobs=-1
)

# Fitting the grid search to the data
lr_search.fit(X_train, y_train)

print("Best C:   ", lr_search.best_params_)
print("Best AUC: ", lr_search.best_score_.round(3))

# Step 3: Evaluate on the test set
best_lr = lr_search.best_estimator_

Best C:    {'model__C': 0.6}
Best AUC:  0.996
